We start by making a connection with Airport database.

In [2]:
import mysql.connector
from mysql.connector import Error
import prettytable

#Creating a connection with Employees database

def create_connection(host_name, user_name, user_password):
    connection = None
    try:
        connection = mysql.connector.connect(
            host=host_name,
            user=user_name,
            passwd=user_password,
            database='airportdb'
        )
        print("Connection to MySQL DB successful")
    except Error as e:
        print(f"The error '{e}' occurred")

    return connection

connection = create_connection("localhost", "root", "kurssql11")
my_cursor=connection.cursor()  


Connection to MySQL DB successful


During preliminary data expolarion, I identified 3 corrupted rows in airport_geo.

In [13]:
my_cursor.execute("""
SELECT
    airport_id,
    name,
    city,
    country,
    latitude,
    longitude
FROM airport_geo
WHERE city=''
""")

my_result=my_cursor.fetchall() #Uploading data
headers = [i[0] for i in my_cursor.description] #Uploading headers
x = prettytable.PrettyTable(headers)

for v in my_result:
    x.add_row(v)

print(x)

+------------+---------------------+------+------------------+----------+-----------+
| airport_id |         name        | city |     country      | latitude | longitude |
+------------+---------------------+------+------------------+----------+-----------+
|    6993    | LITTLE GOOSE LOCK & |      | N463460 W1180004 |   0E-8   |    0E-8   |
|   10037    |         R &         |      | N464958 W1230529 |   0E-8   |    0E-8   |
|   11580    |     ST FRANCIS &    |      | N405901 W0802117 |   0E-8   |    0E-8   |
+------------+---------------------+------+------------------+----------+-----------+


I've managed, based on fragments of names, coordinates (which can be found in "country" column), icao and information available online,
to reconstruct this data.

In [ ]:

my_cursor.execute("""
UPDATE airport_geo
SET name = 'LITTLE GOOSE LOCK AND DAM AIRPORT',
city = 'STARBUCK',
country='UNITED STATES',
latitude='46.5839444',
longitude='118.0035833'
WHERE airport_id = 6993;
-- Source of data: http://airnav.com/airport/16W

UPDATE airport_geo
SET name = UPPER('Sky Ranch Airport'),
city = UPPER('Knoxville'),
country='UNITED STATES',
latitude='35.8856381',
longitude='83.9576836'
WHERE airport_id = 10037;
-- Source of data: https://www.airnav.com/airport/TN98


UPDATE airport_geo
SET name = UPPER('UPMC Jameson Heliport'),
city = UPPER('New Castle'),
country='UNITED STATES',
latitude='41.0126861',
longitude='80.3517111'
WHERE airport_id = 11580;
-- Source of data: https://airnav.com/airport/3PN4
""")


In [8]:
my_cursor.execute("""
SELECT 
	airport_id,
    name,
    city,
    country,
    latitude,
    longitude
FROM airport_geo
WHERE airport_id IN (6993,10037,11580)
""")

my_result=my_cursor.fetchall() #Uploading data
headers = [i[0] for i in my_cursor.description] #Uploading headers
x = prettytable.PrettyTable(headers)

for v in my_result:
    x.add_row(v)

print(x)

+------------+-----------------------------------+------------+---------------+-------------+--------------+
| airport_id |                name               |    city    |    country    |   latitude  |  longitude   |
+------------+-----------------------------------+------------+---------------+-------------+--------------+
|    6993    | LITTLE GOOSE LOCK AND DAM AIRPORT |  STARBUCK  | UNITED STATES | 46.58394440 | 118.00358330 |
|   10037    |         SKY RANCH AIRPORT         | KNOXVILLE  | UNITED STATES | 35.88563810 | 83.95768360  |
|   11580    |       UPMC JAMESON HELIPORT       | NEW CASTLE | UNITED STATES | 41.01268610 | 80.35171110  |
+------------+-----------------------------------+------------+---------------+-------------+--------------+


We start by finding 3 most popular routes.

In [8]:
my_cursor.execute("""
WITH routes AS
			(
			SELECT
				f.flight_id,
				f.from,
				ag1.city AS city1,
				ag1.country AS country1,
				f.to,
				ag2.city AS city2,
				ag2.country AS country2
			FROM flight f
			LEFT JOIN airport_geo ag1
				ON ag1.airport_id=f.from
			LEFT JOIN airport_geo ag2
				ON ag2.airport_id=f.to
			)
		
SELECT
    concat(city1,'(',country1,')','->',city2,'(',country2,')') AS route,
    count(flight_id) AS number_of_flights
FROM routes
GROUP BY route
ORDER BY number_of_flights DESC
LIMIT 3
""")

my_result=my_cursor.fetchall() #Uploading data
headers = [i[0] for i in my_cursor.description] #Uploading headers
x = prettytable.PrettyTable(headers)

for v in my_result:
    x.add_row(v)

print(x)

+------------------------------------------+-------------------+
|                  route                   | number_of_flights |
+------------------------------------------+-------------------+
|   EL SALVADOR(CHILE)->VILSECK(GERMANY)   |        119        |
| CHILECITO(ARGENTINA)->ST ANTHONY(CANADA) |         94        |
|     SINGKEP(INDONESIA)->VAN(TURKEY)      |         93        |
+------------------------------------------+-------------------+


For each airport we find the most popular destination. 
We present first 10 airports (with respect to higest number of flights) with their most popular destination.

In [12]:
my_cursor.execute("""
WITH city_and_country AS
			(
			SELECT
				airport_id,
				concat(city,' (',country,')') AS city_country
			FROM airport_geo
			),
            
dest as
			(
			SELECT
				f.from AS departure,
				c.city_country AS destination,
				count(*) AS number
			FROM flight f
			JOIN city_and_country c
				ON c.airport_id=f.to
			GROUP BY f.from, c.city_country
			),


ranked AS
			(
			SELECT 
				departure,
				destination,
				number,
				DENSE_RANK() OVER (PARTITION BY departure ORDER BY number DESC) as rnk
			FROM dest
			)
            
SELECT
	a.name as airport_name,
    r.destination,
    r.number as number_of_flights
FROM ranked r
JOIN airport_geo a
	ON a.airport_id=r.departure
WHERE rnk=1
ORDER BY number_of_flights DESC, name
LIMIT 10
""")

my_result=my_cursor.fetchall() #Uploading data
headers = [i[0] for i in my_cursor.description] #Uploading headers
x = prettytable.PrettyTable(headers)

for v in my_result:
    x.add_row(v)

print(x)

+--------------------------------+------------------------+-------------------+
|          airport_name          |      destination       | number_of_flights |
+--------------------------------+------------------------+-------------------+
|        EL SALVADOR BAJO        |   VILSECK (GERMANY)    |        119        |
|           CHILECITO            |  ST ANTHONY (CANADA)   |         94        |
|           AKTYUBINSK           |    GISENYI (RWANDA)    |         93        |
|            AL BAHA             |   WLOCLAWEK (POLAND)   |         93        |
| ALFEREZ DAVID FIGUEROA FERNAN* |    SAVERNE (FRANCE)    |         93        |
|            AMARAIS             |    LE LUC (FRANCE)     |         93        |
|          ARTESIA MUN           |   WADI HALFA (SUDAN)   |         93        |
|             AVALON             | HRADEC KRALOVE (CZECH) |         93        |
|       BAD FRANKENHAUSEN        |     LUEBO (CONGO)      |         93        |
|         BAR-SUR-SEINE          | SCHWE

Next we want to find 10 airports with the higest number of destination countries.

In [3]:
my_cursor.execute("""
SELECT
	ag1.name AS name,
    ag1.city,
    ag1.country,
    COUNT(DISTINCT ag2.country) AS number_of_countries
FROM flight f
JOIN airport_geo ag1
	ON ag1.airport_id=f.from
JOIN airport_geo ag2
	ON ag2.airport_id=f.to
GROUP BY ag1.airport_id, name, city, country
ORDER BY number_of_countries DESC, name
LIMIT 10
""")

my_result=my_cursor.fetchall() #Uploading data
headers = [i[0] for i in my_cursor.description] #Uploading headers
x = prettytable.PrettyTable(headers)

for v in my_result:
    x.add_row(v)

print(x)

+-------------------+-------------+------------+---------------------+
|        name       |     city    |  country   | number_of_countries |
+-------------------+-------------+------------+---------------------+
|       BUTARE      |    BUTARE   |   RWANDA   |          7          |
|       ARMOR       |  ST BRIEUC  |   FRANCE   |          6          |
|     ARRACHART     | ANTSIRANANA | MADAGASCAR |          6          |
| BORG EL ARAB INTL |  ALEXANDRIA |   EGYPT    |          6          |
|  R DETOMASI INTL  |   MERCEDES  |  URUGUAY   |          6          |
|        TUPA       |     TUPA    |   BRAZIL   |          6          |
|     VAERNES AB    |  TRONDHEIM  |   NORWAY   |          6          |
|      AEROCLUB     |   ARACAJU   |   BRAZIL   |          5          |
|    BERMUDA INTL   |  BERMUDA IS |  BERMUDA   |          5          |
|      COLTINES     |   ST FLOUR  |   FRANCE   |          5          |
+-------------------+-------------+------------+---------------------+


Next we find 3 flights with the lowest occupancy rate.

In [3]:
my_cursor.execute("""
WITH seats AS
			(
			SELECT
				flight_id,
				count(*) AS sold_seats
			FROM booking
			GROUP BY flight_id
			),
            
city_and_country AS
			(
			SELECT
				airport_id,
				concat(city,' (',country,')') AS city_country
			FROM airport_geo
			),
            
            
flight_info AS
			(
			SELECT
				f.flight_id,
                a.capacity,
				DATE(f.departure) AS departure_date,
				c1.city_country AS departure,
				c2.city_country AS arrival
			FROM flight f
			JOIN  city_and_country c1
				ON c1.airport_id=f.from
			JOIN  city_and_country c2
				ON c2.airport_id=f.to
			JOIN airplane a
				ON a.airplane_id=f.airplane_id
			 )

SELECT
	s.flight_id,
	f.capacity,
    COALESCE(s.sold_seats, 0) as sold_seats,
    ROUND((COALESCE(s.sold_seats, 0)/ f.capacity)*100,2) as occupancy_rate,
    f.departure_date,
    f.departure,
    f.arrival
FROM flight_info f
LEFT JOIN seats as s
	on f.flight_id=s.flight_id
ORDER BY occupancy_rate
LIMIT 3
""")

my_result=my_cursor.fetchall() #Uploading data
headers = [i[0] for i in my_cursor.description] #Uploading headers
x = prettytable.PrettyTable(headers)

for v in my_result:
    x.add_row(v)

print(x)

+-----------+----------+------------+----------------+----------------+------------------------------+----------------------+
| flight_id | capacity | sold_seats | occupancy_rate | departure_date |          departure           |       arrival        |
+-----------+----------+------------+----------------+----------------+------------------------------+----------------------+
|   420892  |    50    |     11     |     22.00      |   2015-07-22   | HERTEN-RHEINFELDEN (GERMANY) | TANQUE NOVO (BRAZIL) |
|   173807  |    50    |     11     |     22.00      |   2015-06-22   |     CHARLEROI (BELGIUM)      |  LILONGWE (MALAWI)   |
|   689843  |    50    |     12     |     24.00      |   2015-08-24   |  WOLF POINT (UNITED STATES)  | ARNHEM (NETHERLANDS) |
+-----------+----------+------------+----------------+----------------+------------------------------+----------------------+


Considering higest occupancy race we can find flights that were overbooked (i.e. there were more sold seats than capacity).

In [5]:
my_cursor.execute("""
WITH seats AS
			(
			SELECT
				flight_id,
				count(*) AS sold_seats
			FROM booking
			GROUP BY flight_id
			),
            
city_and_country AS
			(
			SELECT
				airport_id,
				concat(city,' (',country,')') AS city_country
			FROM airport_geo
			),
            
            
flight_info AS
			(
			SELECT
				f.flight_id,
                a.capacity,
				DATE(f.departure) AS departure_date,
				c1.city_country AS departure,
				c2.city_country AS arrival
			FROM flight f
			JOIN  city_and_country c1
				ON c1.airport_id=f.from
			JOIN  city_and_country c2
				ON c2.airport_id=f.to
			JOIN airplane a
				ON a.airplane_id=f.airplane_id
			 )

SELECT
	s.flight_id,
	f.capacity,
    COALESCE(s.sold_seats, 0) as sold_seats,
    ROUND((COALESCE(s.sold_seats, 0)/ f.capacity)*100,2) as occupancy_rate,
    f.departure_date,
    f.departure,
    f.arrival
FROM flight_info f
LEFT JOIN seats as s
	on f.flight_id=s.flight_id
WHERE COALESCE(s.sold_seats, 0) > capacity
ORDER BY occupancy_rate desc
""")

my_result=my_cursor.fetchall() #Uploading data
headers = [i[0] for i in my_cursor.description] #Uploading headers
x = prettytable.PrettyTable(headers)

for v in my_result:
    x.add_row(v)

print(x)

+-----------+----------+------------+----------------+----------------+--------------------------+------------------------+
| flight_id | capacity | sold_seats | occupancy_rate | departure_date |        departure         |        arrival         |
+-----------+----------+------------+----------------+----------------+--------------------------+------------------------+
|   533286  |    50    |     54     |     108.00     |   2015-08-05   |  SANTA MARTA (COLOMBIA)  |   FALKOPING (SWEDEN)   |
|   184856  |    50    |     53     |     106.00     |   2015-06-23   |   BENI-DIBELE (CONGO)    | MARYS HARBOUR (CANADA) |
|   198439  |    50    |     52     |     104.00     |   2015-06-25   |   ALEXANDRIA (CANADA)    |  KHOST (AFGHANISTAN)   |
|   299006  |    50    |     51     |     102.00     |   2015-07-07   | CHEHALIS (UNITED STATES) |    BALBOA (PANAMA)     |
+-----------+----------+------------+----------------+----------------+--------------------------+------------------------+
